In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pathlib import Path


In [2]:
PROJECT_ROOT = Path("/workspaces/spark")


In [3]:
spark = (SparkSession.builder
         .appName("shopflow-silver-orders")
         .master("local[*]")
         .getOrCreate())

26/10/06 09:11:33 WARN Utils: Your hostname, codespaces-155d86 resolves to a loopback address: 127.0.0.1; using 10.0.12.212 instead (on interface eth0)
26/10/06 09:11:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 09:11:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark.sparkContext.setJobDescription("READ bronze orders")
bronze = spark.read.parquet(str(PROJECT_ROOT / "data/bronze/orders"))
bronze.show(5, truncate=False)

+-----------+-----------+--------+-------------------------+----------+--------+---------+--------------------------+----------------------------------------------+--------------+
|channel    |customer_id|order_id|order_ts                 |product_id|quantity|status   |ingestion_ts              |source_file                                   |ingestion_date|
+-----------+-----------+--------+-------------------------+----------+--------+---------+--------------------------+----------------------------------------------+--------------+
|app        |4519       |13968   |2026-07-16 00:08:10.84214|230       |5       |completed|2026-10-06 08:10:14.274813|file:///workspaces/spark/data/raw/orders.jsonl|2026-10-06    |
|web        |2087       |5839    |2026-09-16 19:26:22.84214|175       |5       |completed|2026-10-06 08:10:14.274813|file:///workspaces/spark/data/raw/orders.jsonl|2026-10-06    |
|app        |3080       |42361   |2026-08-03 08:44:22.84214|44        |2       |failed   |2026-10-06

lista ei product_id validi, per il controllo di integrità referenziale


In [5]:
valid_products = (
    spark.read.parquet(str(PROJECT_ROOT / "data/bronze/products"))
    .select("product_id")
)

valid_products.show(5, truncate=False)

+----------+
|product_id|
+----------+
|1         |
|2         |
|3         |
|4         |
|5         |
+----------+
only showing top 5 rows



list comprehension Python standard (non specifica di Spark): itera su ogni Row della lista appena ottenuta da .collect(), estrae solo il valore di product_id da ciascuna, e costruisce una lista Python semplice

In [6]:
valid_product_ids = [row.product_id for row in valid_products.collect()]
print(valid_product_ids[:5])

['1', '2', '3', '4', '5']


In [7]:
# 1. flag di validazione, indipendenti tra loro

checked = (
    bronze
    .withColumn("err_null_customer", F.col("customer_id").isNull())
    .withColumn("err_negative_qty", F.col("quantity") < 0)
    .withColumn("err_invalid_product", ~F.col("product_id").isin(valid_product_ids))
)

checked.show(5, truncate=False)

+-----------+-----------+--------+-------------------------+----------+--------+---------+--------------------------+----------------------------------------------+--------------+-----------------+----------------+-------------------+
|channel    |customer_id|order_id|order_ts                 |product_id|quantity|status   |ingestion_ts              |source_file                                   |ingestion_date|err_null_customer|err_negative_qty|err_invalid_product|
+-----------+-----------+--------+-------------------------+----------+--------+---------+--------------------------+----------------------------------------------+--------------+-----------------+----------------+-------------------+
|app        |4519       |13968   |2026-07-16 00:08:10.84214|230       |5       |completed|2026-10-06 08:10:14.274813|file:///workspaces/spark/data/raw/orders.jsonl|2026-10-06    |false            |false           |false              |
|web        |2087       |5839    |2026-09-16 19:26:22.84214|

2. deduplicazione via window function

In [8]:
w = Window.partitionBy("order_id").orderBy(F.col("order_ts").asc())
ranked = checked.withColumn("rn", F.row_number().over(w))

ranked.show(5, truncate=False)

+-----------+-----------+--------+-------------------------+----------+--------+---------+--------------------------+----------------------------------------------+--------------+-----------------+----------------+-------------------+---+
|channel    |customer_id|order_id|order_ts                 |product_id|quantity|status   |ingestion_ts              |source_file                                   |ingestion_date|err_null_customer|err_negative_qty|err_invalid_product|rn |
+-----------+-----------+--------+-------------------------+----------+--------+---------+--------------------------+----------------------------------------------+--------------+-----------------+----------------+-------------------+---+
|marketplace|4422       |100     |2026-08-26 15:07:49.84214|270       |1       |completed|2026-10-06 08:10:14.274813|file:///workspaces/spark/data/raw/orders.jsonl|2026-10-06    |false            |false           |false              |1  |
|web        |1187       |1000    |2026-10-03

Per vedere solo i duplicati (quelli con rn > 1

In [21]:
ranked.filter(F.col("rn") > 1).select("order_id", "order_ts", "rn").show(20, truncate=False)

+--------+-------------------------+---+
|order_id|order_ts                 |rn |
+--------+-------------------------+---+
|12160   |2026-07-19 05:50:42.84214|2  |
|12356   |2026-07-17 04:02:29.84214|2  |
|12848   |2026-07-29 14:41:59.84214|2  |
|12919   |2026-09-27 05:06:31.84214|2  |
|13350   |2026-07-25 07:58:54.84214|2  |
|14872   |2026-09-25 18:57:28.84214|2  |
|15474   |2026-08-11 14:36:51.84214|2  |
|1631    |2026-09-26 21:13:15.84214|2  |
|16398   |2026-09-18 08:50:23.84214|2  |
|16571   |2026-07-19 06:31:17.84214|2  |
|17313   |2026-09-18 07:21:33.84214|2  |
|175     |2026-09-12 09:26:42.84214|2  |
|17976   |2026-08-04 23:30:48.84214|2  |
|18049   |2026-09-20 12:12:24.84214|2  |
|19073   |2026-07-13 15:29:36.84214|2  |
|1925    |2026-08-23 17:01:59.84214|2  |
|2040    |2026-09-06 20:23:21.84214|2  |
|20883   |2026-08-13 05:55:14.84214|2  |
|20922   |2026-08-07 12:29:53.84214|2  |
|20965   |2026-09-19 17:52:59.84214|2  |
+--------+-------------------------+---+
only showing top

definiamo condizioni record con errori (has error) e record duplicati (is duplicated) 

In [12]:
has_error = F.col("err_null_customer") | F.col("err_negative_qty") | F.col("err_invalid_product")
is_duplicate = F.col("rn") > 1

3. split record: errori, duplicati (scartati silenziosamente), puliti + creazione df


In [23]:
duplicate = ranked.filter(is_duplicate)

deduped = ranked.filter(F.col("rn") == 1).drop("rn") 
errors = deduped.filter(has_error)
clean = deduped.filter(~has_error)


In [24]:
print("duplicate")
duplicate.show(5, truncate=False)
print(f"Count: {duplicate.count()}")


duplicate
+-----------+-----------+--------+-------------------------+----------+--------+---------+--------------------------+----------------------------------------------+--------------+-----------------+----------------+-------------------+---+
|channel    |customer_id|order_id|order_ts                 |product_id|quantity|status   |ingestion_ts              |source_file                                   |ingestion_date|err_null_customer|err_negative_qty|err_invalid_product|rn |
+-----------+-----------+--------+-------------------------+----------+--------+---------+--------------------------+----------------------------------------------+--------------+-----------------+----------------+-------------------+---+
|app        |4320       |12160   |2026-07-19 05:50:42.84214|78        |1       |completed|2026-10-06 08:10:14.274813|file:///workspaces/spark/data/raw/orders.jsonl|2026-10-06    |false            |false           |false              |2  |
|marketplace|4101       |12356   |

In [25]:
print("errors")
errors.show(5, truncate=False)
print(f"Count: {errors.count()}")

errors
+-------+-----------+--------+-------------------------+----------+--------+---------+--------------------------+----------------------------------------------+--------------+-----------------+----------------+-------------------+
|channel|customer_id|order_id|order_ts                 |product_id|quantity|status   |ingestion_ts              |source_file                                   |ingestion_date|err_null_customer|err_negative_qty|err_invalid_product|
+-------+-----------+--------+-------------------------+----------+--------+---------+--------------------------+----------------------------------------------+--------------+-----------------+----------------+-------------------+
|web    |3645       |10134   |2026-07-11 00:54:38.84214|318       |4       |completed|2026-10-06 08:10:14.274813|file:///workspaces/spark/data/raw/orders.jsonl|2026-10-06    |false            |false           |true               |
|app    |NULL       |10145   |2026-08-20 20:33:20.84214|49        |2 

In [26]:

print("clean")
clean.show(5, truncate=False)
print(f"Count: {clean.count()}")

clean
+-----------+-----------+--------+-------------------------+----------+--------+---------+--------------------------+----------------------------------------------+--------------+-----------------+----------------+-------------------+
|channel    |customer_id|order_id|order_ts                 |product_id|quantity|status   |ingestion_ts              |source_file                                   |ingestion_date|err_null_customer|err_negative_qty|err_invalid_product|
+-----------+-----------+--------+-------------------------+----------+--------+---------+--------------------------+----------------------------------------------+--------------+-----------------+----------------+-------------------+
|marketplace|4422       |100     |2026-08-26 15:07:49.84214|270       |1       |completed|2026-10-06 08:10:14.274813|file:///workspaces/spark/data/raw/orders.jsonl|2026-10-06    |false            |false           |false              |
|web        |1187       |1000    |2026-10-03 10:47:50.

In [27]:
spark.sparkContext.setJobDescription("WRITE silver orders")
(
    clean
    .withColumn("order_date", F.to_date("order_ts"))
    .drop("rn", "err_null_customer", "err_negative_qty", "err_invalid_product")
    .coalesce(4)
    .write
    .mode("overwrite")
    .partitionBy("order_date")
    .parquet(str(PROJECT_ROOT / "data/silver/orders"))
)


In [28]:
spark.sparkContext.setJobDescription("WRITE errors orders")
(
    errors
    .drop("rn")
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(str(PROJECT_ROOT / "data/silver/errors_orders"))
)

In [29]:
spark.stop()